# 🧬 BiomeX — Pipeline Complet : Supervisé + Non-Supervisé + Recommandations

> **Version 5.0 — Février 2026**  
> Abdou Bâ (CTO) · Mouhammadou Dia (CEO) · www.biomex.ai

---

## 📋 Table des matières

### PARTIE A — DONNÉES & PREPROCESSING
- [1. Installation & Imports](#1)
- [2. Simulation & Chargement des données](#2)
- [3. Pipeline bioinformatique (FastQC → HUMAnN3)](#3)
- [4. Matrice globale & Normalisation](#4)
- [5. Réduction de dimension (PCA, UMAP, Autoencoder)](#5)

### PARTIE B — SCORES BIOLOGIQUES
- [6. BiomeX Score Engine (Shannon, Inflam, SCFA, Dysbiose, Global)](#6)

### PARTIE C — APPRENTISSAGE NON SUPERVISÉ
- [7. Clustering K-Means — Profils Microbiomiques](#7)
- [8. Clustering Hiérarchique (Ward)](#8)
- [9. Modèles de Mélange Gaussien (GMM)](#9)
- [10. DBSCAN — Détection d'Anomalies Microbiomiques](#10)
- [11. Analyse de Co-abondance (SparCC / Corrélation)](#11)

### PARTIE D — APPRENTISSAGE SUPERVISÉ
- [12. Régression Logistique · Random Forest · XGBoost](#12)
- [13. Validation croisée & Métriques (AUC, RMSE, SHAP)](#13)

### PARTIE E — MOTEUR DE RECOMMANDATIONS
- [14. Recommandations basées sur les clusters (non supervisé)](#14)
- [15. Filtrage Collaboratif — Patients similaires](#15)
- [16. Système de règles cliniques expertes](#16)
- [17. Moteur de recommandations unifié BiomeX](#17)

### PARTIE F — RAPPORT
- [18. Rapport patient complet + Sauvegarde modèles](#18)

---
## 1. Installation & Imports <a id='1'></a>

In [ ]:
# !pip install numpy pandas scikit-learn xgboost shap matplotlib seaborn
# !pip install torch umap-learn scipy statsmodels tqdm kneed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import braycurtis, pdist, squareform
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn Supervisé ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                      train_test_split, GridSearchCV)
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, mean_squared_error, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

# ── Sklearn Non Supervisé ─────────────────────────────────────────────
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ── Autres ───────────────────────────────────────────────────────────
import xgboost as xgb
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠️  UMAP non installé — pip install umap-learn")

# ── Config visuelle ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

C = {  # Palette BiomeX
    'navy':   '#003366', 'blue':   '#0066CC', 'green':  '#009944',
    'orange': '#E07B00', 'teal':   '#007B8A', 'purple': '#6B3FA0',
    'red':    '#CC0000', 'light':  '#D6E8FA', 'gray':   '#F5F5F5',
}
CLUSTER_PALETTE = ['#0066CC','#009944','#E07B00','#6B3FA0','#007B8A','#CC0000','#FFB300']

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Environnement BiomeX v5.0 prêt")

---
## 2. Chargement des données réelles <a id='2'></a>

In [ ]:
N_PATIENTS = 500
N_BACTERIA = 200
N_FUNCTIONS = 300

BACTERIA_NAMES = (
    ['Lactobacillus_rhamnosus','Lactobacillus_acidophilus','Lactobacillus_fermentum',
     'Lactobacillus_plantarum','Lactobacillus_gasseri',
     'Bifidobacterium_longum','Bifidobacterium_adolescentis','Bifidobacterium_breve',
     'Faecalibacterium_prausnitzii','Roseburia_intestinalis','Ruminococcus_bromii',
     'Clostridium_butyricum','Butyrivibrio_fibrisolvens','Eubacterium_hallii',
     'Akkermansia_muciniphila','Christensenella_minuta',
     'Bacteroides_fragilis','Bacteroides_thetaiotaomicron','Bacteroides_uniformis',
     'Prevotella_copri','Prevotella_stercorea','Prevotella_africana',
     'Alistipes_putredinis','Parabacteroides_distasonis',
     'Escherichia_coli','Klebsiella_pneumoniae','Helicobacter_pylori',
     'Enterobacter_cloacae','Citrobacter_freundii','Ruminococcus_gnavus',
     'Treponema_succinifaciens','Spirochaeta_africana',
     'Streptococcus_salivarius','Streptococcus_thermophilus']
    + [f'Bacterium_sp_{i}' for i in range(1, N_BACTERIA - 34)]
)[:N_BACTERIA]

PATHOBIONTES      = ['Escherichia_coli','Klebsiella_pneumoniae','Helicobacter_pylori',
                     'Enterobacter_cloacae','Citrobacter_freundii','Ruminococcus_gnavus']
BUTYRATE_PRODUCERS= ['Faecalibacterium_prausnitzii','Roseburia_intestinalis',
                     'Clostridium_butyricum','Butyrivibrio_fibrisolvens','Eubacterium_hallii']
BENEFICIAL        = ['Lactobacillus_rhamnosus','Lactobacillus_acidophilus','Lactobacillus_plantarum',
                     'Bifidobacterium_longum','Bifidobacterium_adolescentis',
                     'Akkermansia_muciniphila','Faecalibacterium_prausnitzii']

# ── Simulateurs ───────────────────────────────────────────────────────
def simulate_microbiome(n, bacteria, seed=SEED):
    rng = np.random.RandomState(seed)
    alpha = rng.exponential(0.5, len(bacteria)) + 0.01
    raw = rng.dirichlet(alpha, size=n)
    raw = np.clip(raw + rng.normal(0, 0.003, raw.shape), 0, None)
    raw /= raw.sum(axis=1, keepdims=True)
    depths = rng.randint(50_000, 200_000, n)
    counts = (raw * depths[:, None]).astype(int)
    df = pd.DataFrame(counts, columns=bacteria,
                      index=[f'P{i:04d}' for i in range(n)])
    return df

def simulate_functions(n, n_func, seed=SEED):
    rng = np.random.RandomState(seed)
    cols = ([f'KEGG_K{i:05d}' for i in range(n_func//3)] +
            [f'MetaCyc_{i}'   for i in range(n_func//3)] +
            [f'GO_{i}'        for i in range(n_func - 2*(n_func//3))])
    data = rng.lognormal(2, 1.5, (n, n_func)).clip(0)
    return pd.DataFrame(data, columns=cols, index=[f'P{i:04d}' for i in range(n)])

def simulate_clinical(n, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = [f'P{i:04d}' for i in range(n)]
    df = pd.DataFrame(index=idx)
    df['age']          = rng.randint(18, 75, n)
    df['sexe']         = rng.choice([0,1], n)
    df['IMC']          = rng.normal(26, 5, n).clip(15, 50)
    df['glycemie']     = rng.normal(5.5, 1.2, n).clip(3, 15)
    df['HbA1c']        = rng.normal(5.8, 0.9, n).clip(4, 12)
    df['CRP']          = rng.exponential(3, n).clip(0, 50)
    df['tension_sys']  = rng.normal(125, 18, n).clip(90, 200)
    df['cholesterol']  = rng.normal(4.8, 1.2, n).clip(2, 10)
    df['triglycerides']= rng.exponential(1.5, n).clip(0.3, 10)
    df['ballonnements']= rng.randint(0, 4, n)
    df['transit']      = rng.randint(0, 4, n)
    df['fatigue']      = rng.randint(0, 4, n)
    df['ethnie']       = rng.choice(['Wolof','Sérère','Peul','Diola','Mandingue'], n)
    df['zone']         = rng.choice(['Urbain','Banlieue','Rural'], n, p=[0.5,0.35,0.15])
    df['antibio_3mois']= rng.choice([0,1], n, p=[0.75,0.25])
    df['fibers_g']     = rng.normal(22, 8, n).clip(5, 60)
    df['sugar_g']      = rng.normal(60, 25, n).clip(5, 200)
    df['millet_freq']  = rng.randint(0, 8, n)
    df['niebe_freq']   = rng.randint(0, 8, n)
    df['fish_portions']= rng.randint(0, 7, n)
    # Labels
    df['diabete']      = ((df['glycemie']>7.0)|(df['HbA1c']>6.5)).astype(int)
    df['metabolique']  = ((df['IMC']>30)&(df['glycemie']>6.1)).astype(int)
    df['inflam_elev']  = (df['CRP']>10).astype(int)
    return df

print("🔬 Chargement des données...")
from pathlib import Path


def normalize_species_name(name):
    return (
        str(name)
        .strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "_")
    )


def load_real_viome_data(data_root, max_bacteria=200, max_functions=300):
    metadata_path = data_root / "Viome_microbiome_metadata.csv"
    taxa_path = data_root / "viome-abundance" / "abundance" / "Viome_species_readcount_40samples.csv"
    func_path = data_root / "viome-abundance" / "abundance" / "Viome_function_KO_readcount_40samples.csv"

    if not metadata_path.exists() or not taxa_path.exists() or not func_path.exists():
        missing = [str(p) for p in [metadata_path, taxa_path, func_path] if not p.exists()]
        raise FileNotFoundError(f"Fichiers manquants: {missing}")

    taxa_long = pd.read_csv(taxa_path)
    taxa_long['species_name_norm'] = taxa_long['species_name'].map(normalize_species_name)
    X_taxa_all = taxa_long.pivot_table(
        index='sample_name',
        columns='species_name_norm',
        values='species_readcount',
        aggfunc='sum',
        fill_value=0,
    ).sort_index()

    priority_species = [
        'Escherichia_coli', 'Klebsiella_pneumoniae', 'Helicobacter_pylori',
        'Enterobacter_cloacae', 'Citrobacter_freundii', 'Ruminococcus_gnavus',
        'Faecalibacterium_prausnitzii', 'Roseburia_intestinalis',
        'Clostridium_butyricum', 'Butyrivibrio_fibrisolvens', 'Eubacterium_hallii',
        'Lactobacillus_rhamnosus', 'Lactobacillus_acidophilus', 'Lactobacillus_plantarum',
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis', 'Akkermansia_muciniphila',
    ]
    priority_species_present = [s for s in priority_species if s in X_taxa_all.columns]
    top_species = X_taxa_all.sum(axis=0).sort_values(ascending=False).head(max_bacteria).index.tolist()
    selected_species = list(dict.fromkeys(priority_species_present + top_species))[:max_bacteria]
    X_taxa = X_taxa_all[selected_species].copy()

    func_long = pd.read_csv(func_path)
    X_func_all = func_long.pivot_table(
        index='sample_name',
        columns='function_name',
        values='readcount',
        aggfunc='sum',
        fill_value=0,
    ).sort_index()
    top_functions = X_func_all.sum(axis=0).sort_values(ascending=False).head(max_functions).index.tolist()
    X_func = X_func_all[top_functions].copy()

    meta = pd.read_csv(metadata_path)
    symptom_t2_col = 'symptome_T2' if 'symptome_T2' in meta.columns else 'symptom_T2'

    meta_t1 = meta[['subject_id', 'age', 'gender', 'bmi', 'symptom_T1', 'sample_name_T1']].rename(
        columns={'symptom_T1': 'symptom', 'sample_name_T1': 'sample_name'}
    )
    meta_t2 = meta[['subject_id', 'age', 'gender', 'bmi', symptom_t2_col, 'sample_name_T2']].rename(
        columns={symptom_t2_col: 'symptom', 'sample_name_T2': 'sample_name'}
    )

    clinical_map = pd.concat([meta_t1, meta_t2], ignore_index=True)
    clinical_map = clinical_map.dropna(subset=['sample_name'])
    clinical_map['sample_name'] = clinical_map['sample_name'].astype(str).str.strip()
    clinical_map = clinical_map.drop_duplicates(subset=['sample_name'], keep='last').set_index('sample_name')

    common_samples = sorted(set(X_taxa.index) & set(X_func.index) & set(clinical_map.index))
    if len(common_samples) == 0:
        raise ValueError("Aucun sample commun entre taxonomie, fonctions et metadata")

    return (
        X_taxa.loc[common_samples].copy(),
        X_func.loc[common_samples].copy(),
        clinical_map.loc[common_samples].copy(),
    )


def build_clinical_from_metadata(clinical_map, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = clinical_map.index
    n = len(idx)

    age = pd.to_numeric(clinical_map['age'], errors='coerce').clip(18, 90)
    if age.isna().any():
        age = age.fillna(age.median() if not age.dropna().empty else 35)

    imc = pd.to_numeric(clinical_map['bmi'], errors='coerce').clip(15, 50)
    if imc.isna().any():
        imc = imc.fillna(imc.median() if not imc.dropna().empty else 26)

    gender = clinical_map['gender'].fillna('').astype(str).str.upper().str.strip().str[0]
    symptom = clinical_map['symptom'].fillna('Moderate').astype(str).str.lower().str.strip()
    symptom_score = symptom.map({'none': 0, 'mild': 1, 'moderate': 2, 'severe': 3}).fillna(1).astype(int)

    df = pd.DataFrame(index=idx)
    df['age'] = age.round(2)
    df['sexe'] = gender.map({'F': 0, 'M': 1}).fillna(pd.Series(rng.choice([0, 1], n), index=idx)).astype(int)
    df['IMC'] = imc.round(2)

    df['glycemie'] = np.clip(4.6 + 0.045 * (df['IMC'] - 22) + 0.012 * (df['age'] - 30) + rng.normal(0, 0.6, n), 3, 15)
    df['HbA1c'] = np.clip(4.8 + 0.33 * (df['glycemie'] - 5.0) + rng.normal(0, 0.35, n), 4, 12)
    df['CRP'] = np.clip(rng.exponential(2.8, n) + symptom_score.values * 0.8, 0, 50)
    df['tension_sys'] = np.clip(110 + 0.65 * df['age'] + 0.9 * (df['IMC'] - 23) + rng.normal(0, 7, n), 90, 200)
    df['cholesterol'] = np.clip(3.8 + 0.06 * (df['IMC'] - 22) + rng.normal(0, 0.55, n), 2, 10)
    df['triglycerides'] = np.clip(rng.exponential(1.25, n) + 0.03 * (df['IMC'] - 20), 0.3, 10)

    df['ballonnements'] = symptom_score.values
    df['transit'] = np.where(
        symptom_score.values >= 2,
        rng.choice([1, 2, 3], n, p=[0.35, 0.45, 0.20]),
        rng.choice([0, 1, 2], n, p=[0.70, 0.20, 0.10]),
    ).astype(int)
    df['fatigue'] = np.clip(symptom_score.values + rng.randint(0, 2, n), 0, 3).astype(int)

    df['ethnie'] = 'NA'
    df['zone'] = rng.choice(['Urbain', 'Banlieue', 'Rural'], n, p=[0.5, 0.35, 0.15])
    df['antibio_3mois'] = rng.choice([0, 1], n, p=[0.75, 0.25])
    df['fibers_g'] = np.clip(26 - symptom_score.values * 2 + rng.normal(0, 4, n), 5, 60)
    df['sugar_g'] = np.clip(45 + symptom_score.values * 8 + rng.normal(0, 12, n), 5, 200)
    df['millet_freq'] = rng.randint(0, 8, n)
    df['niebe_freq'] = rng.randint(0, 8, n)
    df['fish_portions'] = rng.randint(0, 7, n)

    df['diabete'] = ((df['glycemie'] > 7.0) | (df['HbA1c'] > 6.5)).astype(int)
    df['metabolique'] = ((df['IMC'] > 30) & (df['glycemie'] > 6.1)).astype(int)
    df['inflam_elev'] = (df['CRP'] > 10).astype(int)
    return df


data_candidates = [
    Path.cwd() / "Data" / "Data_proccesing",
    Path.cwd().parent / "Data" / "Data_proccesing",
    Path("/home/cardan/Documents/BiomeX_projet/Data/Data_proccesing"),
]
data_root = next((p for p in data_candidates if p.exists()), None)

if data_root is not None:
    try:
        X_taxa, X_func, clinical_map = load_real_viome_data(data_root)
        df_clin = build_clinical_from_metadata(clinical_map)
        BACTERIA_NAMES = list(X_taxa.columns)

        base_pathobiontes = [
            'Escherichia_coli', 'Klebsiella_pneumoniae', 'Helicobacter_pylori',
            'Enterobacter_cloacae', 'Citrobacter_freundii', 'Ruminococcus_gnavus'
        ]
        base_butyrate = [
            'Faecalibacterium_prausnitzii', 'Roseburia_intestinalis',
            'Clostridium_butyricum', 'Butyrivibrio_fibrisolvens', 'Eubacterium_hallii'
        ]
        base_beneficial = [
            'Lactobacillus_rhamnosus', 'Lactobacillus_acidophilus', 'Lactobacillus_plantarum',
            'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
            'Akkermansia_muciniphila', 'Faecalibacterium_prausnitzii'
        ]

        PATHOBIONTES = [b for b in base_pathobiontes if b in BACTERIA_NAMES]
        BUTYRATE_PRODUCERS = [b for b in base_butyrate if b in BACTERIA_NAMES]
        BENEFICIAL = [b for b in base_beneficial if b in BACTERIA_NAMES]

        print(f"✅ Données réelles chargées depuis: {data_root}")
    except Exception as e:
        print(f"⚠️ Chargement réel impossible ({e}) — fallback simulation")
        X_taxa = simulate_microbiome(N_PATIENTS, BACTERIA_NAMES)
        X_func = simulate_functions(N_PATIENTS, N_FUNCTIONS)
        df_clin = simulate_clinical(N_PATIENTS)
else:
    print("⚠️ Dossier Data/Data_proccesing introuvable — fallback simulation")
    X_taxa = simulate_microbiome(N_PATIENTS, BACTERIA_NAMES)
    X_func = simulate_functions(N_PATIENTS, N_FUNCTIONS)
    df_clin = simulate_clinical(N_PATIENTS)

# Mise à jour des dimensions selon les données réellement chargées
N_PATIENTS = X_taxa.shape[0]
N_BACTERIA = X_taxa.shape[1]
N_FUNCTIONS = X_func.shape[1]

# Abondances relatives
row_sums = X_taxa.sum(axis=1)
X_rel = X_taxa.div(row_sums, axis=0)

# CLR (Centered Log-Ratio) — standard microbiome
X_pseudo = X_rel + 1e-6
X_log = np.log(X_pseudo)
X_clr = pd.DataFrame(
    X_log.values - X_log.mean(axis=1).values[:, None],
    index=X_rel.index, columns=X_rel.columns
)

print(f"  ✅ Taxa       : {X_taxa.shape}")
print(f"  ✅ Fonctions  : {X_func.shape}")
print(f"  ✅ Clinique   : {df_clin.shape}")


---
## 3. Pipeline Bioinformatique <a id='3'></a>

In [ ]:
# ── Wrappers pipeline (appelés en production via subprocess) ──────────

PIPELINE_STEPS = [
    # (Numéro, Outil, Description, Commande réelle)
    (1, 'FastQC',      'Contrôle qualité reads bruts',
     'fastqc {input} -o {out} --threads 8'),
    (2, 'Trimmomatic', 'Nettoyage adaptateurs & qualité',
     'trimmomatic PE {R1} {R2} ... LEADING:20 TRAILING:20 MINLEN:50'),
    (3, 'MEGAHIT',     'Assemblage métagénomique de novo',
     'megahit -1 {R1} -2 {R2} -o {out} --min-contig-len 500'),
    (4, 'Kraken2',     'Classification taxonomique',
     'kraken2 --db {db} --paired --report report.txt {R1} {R2}'),
    (5, 'HUMAnN3',     'Annotation fonctionnelle KEGG/MetaCyc/GO',
     'humann3 --input {input} --output {out} --threads 16'),
    (6, 'Bracken',     'Réestimation abondances au niveau espèce',
     'bracken -d {db} -i {report} -o {out} -r 150 -l S'),
    (7, 'QIIME2',      '(16S) Débruitage DADA2 + table ASV',
     'qiime dada2 denoise-paired --p-trim-left-f 0 --p-trunc-len-f 250'),
]

print("\n🔬 PIPELINE BIOINFORMATIQUE BIOMEX\n" + "─"*55)
for num, tool, desc, cmd in PIPELINE_STEPS:
    print(f"  [{num}] {tool:<14} → {desc}")
print("─"*55)
print("  Output final : matrice X_taxa (Patients × Espèces)")
print("                 matrice X_func  (Patients × Fonctions)")

# Simulation d'un rapport FastQC
def run_pipeline_report(n=5):
    rng = np.random.RandomState(SEED)
    samples = []
    for i in range(n):
        total   = rng.randint(800_000, 3_000_000)
        kept    = int(total * rng.uniform(0.88, 0.98))
        contigs = rng.randint(50_000, 250_000)
        samples.append({
            'Sample'       : f'P{i:04d}',
            'Raw reads'    : f'{total:,}',
            'After trim'   : f'{kept:,}  ({kept/total:.1%})',
            'Phred moyen'  : round(rng.uniform(29, 37), 1),
            'Contigs'      : f'{contigs:,}',
            'N50 (bp)'     : rng.randint(500, 8000),
            'Espèces det.' : rng.randint(80, 180),
            'Voies KEGG'   : rng.randint(120, 260),
        })
    return pd.DataFrame(samples).set_index('Sample')

print("\n📊 Rapport pipeline (5 premiers patients) :")
print(run_pipeline_report().to_string())

---
## 4. Matrice Globale & Normalisation <a id='4'></a>

$$X_{total} = [X_{microbiome} \mid X_{fonctions} \mid X_{clinique}]$$

In [ ]:
X_clin_num = df_clin.select_dtypes(include=np.number).drop(
    columns=['diabete','metabolique','inflam_elev'], errors='ignore')

# Normalisation par bloc
scaler_clin = StandardScaler()
scaler_func = StandardScaler()

X_clin_sc = pd.DataFrame(scaler_clin.fit_transform(X_clin_num),
                          index=X_clin_num.index, columns=X_clin_num.columns)
X_func_sc = pd.DataFrame(scaler_func.fit_transform(X_func),
                          index=X_func.index, columns=X_func.columns)

X_total = pd.concat([X_clr, X_func_sc, X_clin_sc], axis=1)
n, p    = X_total.shape

print(f"✅ X_total = {n} patients × {p} features")
print(f"   CLR taxonomique   : {X_clr.shape[1]} features")
print(f"   Fonctions HUMAnN3 : {X_func_sc.shape[1]} features")
print(f"   Variables cliniques: {X_clin_sc.shape[1]} features")

# Heatmap des corrélations inter-blocs (sous-échantillon)
sample_taxa = X_rel[BACTERIA_NAMES[:15]]
sample_clin = X_clin_num[['glycemie','IMC','CRP','HbA1c','fibers_g','sugar_g']]
corr_cross  = pd.concat([sample_taxa, sample_clin], axis=1).corr().iloc[:15, 15:]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_cross, cmap='RdYlGn', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Corrélation de Spearman'})
ax.set_title('Corrélations Taxons × Biomarqueurs Cliniques',
             fontsize=13, fontweight='bold', color=C['navy'])
plt.tight_layout()
plt.savefig('01_corr_taxa_clin.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Réduction de Dimension <a id='5'></a>

In [ ]:
# ── PCA ───────────────────────────────────────────────────────────────
pca      = PCA(n_components=50, random_state=SEED)
X_pca    = pca.fit_transform(X_clr.fillna(0))
cumvar   = np.cumsum(pca.explained_variance_ratio_)
n_comp90 = int(np.argmax(cumvar >= 0.90) + 1)
X_pca_90 = X_pca[:, :n_comp90]

# ── UMAP ─────────────────────────────────────────────────────────────
if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, n_neighbors=15,
                        min_dist=0.1, metric='euclidean', random_state=SEED)
    X_umap  = reducer.fit_transform(X_pca_90)
else:
    # Fallback : t-SNE via sklearn
    from sklearn.manifold import TSNE
    X_umap = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(X_pca_90)

# ── Autoencoder PyTorch ───────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, in_dim, latent=32, hidden=(256, 128)):
        super().__init__()
        enc, prev = [], in_dim
        for h in hidden:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        enc.append(nn.Linear(prev, latent))
        self.enc = nn.Sequential(*enc)
        dec, prev = [], latent
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        dec.append(nn.Linear(prev, in_dim))
        self.dec = nn.Sequential(*dec)
    def forward(self, x):
        z = self.enc(x); return self.dec(z), z

def train_ae(X_np, latent=32, epochs=50, bs=64, lr=1e-3):
    X_t  = torch.FloatTensor(X_np)
    dl   = DataLoader(TensorDataset(X_t), batch_size=bs, shuffle=True)
    model= Autoencoder(X_np.shape[1], latent)
    opt  = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train(); tot = 0
        for (b,) in dl:
            opt.zero_grad(); xh, _ = model(b)
            loss = crit(xh, b); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        losses.append(tot / len(X_np)); sch.step()
    model.eval()
    with torch.no_grad(): _, Z = model(X_t)
    return model, Z.numpy(), losses

print("🧠 Entraînement Autoencoder...")
scaler_ae = StandardScaler()
X_ae_in   = scaler_ae.fit_transform(X_clr.fillna(0))
ae_model, Z_latent, ae_losses = train_ae(X_ae_in.astype(np.float32),
                                          latent=32, epochs=50)

# ── Visualisation ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Réduction de Dimension — PCA · UMAP · Autoencoder',
             fontsize=14, fontweight='bold', color=C['navy'])

# PCA PC1 vs PC2
for label, col, name in [(0,C['green'],'Faible risque'),(1,C['orange'],'Risque élevé')]:
    m = df_clin['diabete']==label
    axes[0].scatter(X_pca[m,0], X_pca[m,1], c=col, alpha=0.5, s=30,
                    label=name, edgecolors='white', lw=0.3)
axes[0].set_title(f'PCA — PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%) vs PC2')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend()

# UMAP
for label, col, name in [(0,C['green'],'Faible risque'),(1,C['orange'],'Risque élevé')]:
    m = df_clin['diabete']==label
    axes[1].scatter(X_umap[m,0], X_umap[m,1], c=col, alpha=0.5, s=30,
                    label=name, edgecolors='white', lw=0.3)
axes[1].set_title('UMAP 2D'); axes[1].set_xlabel('UMAP1'); axes[1].set_ylabel('UMAP2')
axes[1].legend()

# Autoencoder loss
axes[2].plot(ae_losses, color=C['purple'], lw=2)
axes[2].fill_between(range(len(ae_losses)), ae_losses, alpha=0.2, color=C['purple'])
axes[2].set_title('Autoencoder — Courbe de perte (MSE)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE Loss')

plt.tight_layout()
plt.savefig('02_dim_reduction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ PCA : {n_comp90} composantes (90% variance) · UMAP 2D · Autoencoder latent dim=32")

---
## 6. BiomeX Score Engine <a id='6'></a>

In [ ]:
INFLAM_WEIGHTS = {
    'Escherichia_coli':2.0,'Klebsiella_pneumoniae':1.8,'Helicobacter_pylori':2.5,
    'Enterobacter_cloacae':1.5,'Ruminococcus_gnavus':1.2,'Citrobacter_freundii':1.4,
    'Faecalibacterium_prausnitzii':-3.0,'Lactobacillus_rhamnosus':-2.0,
    'Bifidobacterium_longum':-1.8,'Akkermansia_muciniphila':-2.5,
    'Roseburia_intestinalis':-1.5,'Lactobacillus_plantarum':-1.2,
}

def shannon_H(row):
    p = row[row > 0]
    p = p / p.sum()
    return -float(np.sum(p * np.log(p)))

def inflammation(row):
    return float(sum(w * row.get(b, 0) for b, w in INFLAM_WEIGHTS.items()))

def scfa(row):
    producers = [b for b in BUTYRATE_PRODUCERS if b in row.index]
    return float(row[producers].sum()) if producers else 0.0

def dysbiose(row):
    pa = sum(row.get(b, 0) for b in PATHOBIONTES)
    be = sum(row.get(b, 0) for b in BENEFICIAL)
    return float(pa / (be + 1e-10))

def global_score(H, s, d, f, a=0.3, b=0.3, g=0.2, delta=0.2):
    raw = a*H + b*s - g*d - delta*f
    return float(np.clip((raw + 3) / 6 * 100, 0, 100))

print("⚙️  Calcul des scores biologiques sur la cohorte...")
rows = []
for pid in X_rel.index:
    row = X_rel.loc[pid]
    H   = shannon_H(row)
    f_  = inflammation(row)
    s_  = scfa(row)
    d_  = dysbiose(row)
    g_  = global_score(H, s_, d_, f_)
    rows.append({'pid':pid,'H':H,'inflam':f_,'scfa':s_,'dysbiose':d_,'global':g_})

SC = pd.DataFrame(rows).set_index('pid')
SC['niv_sante'] = pd.cut(SC['global'], bins=[0,33,66,100],
                          labels=['Préoccupant','Modéré','Optimal'])
print("✅ Scores calculés")
print(SC.describe().round(3))

---
## PARTIE C — APPRENTISSAGE NON SUPERVISÉ

---
## 7. Clustering K-Means — Profils Microbiomiques <a id='7'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CHOIX DU K OPTIMAL — Méthode du coude + Silhouette + Davies-Bouldin
# ══════════════════════════════════════════════════════════════════════

# Espace de features pour le clustering : PCA (90%) + Scores biologiques
X_cluster = np.hstack([X_pca_90, SC[['H','inflam','scfa','dysbiose','global']].values])
X_cluster = StandardScaler().fit_transform(X_cluster)

K_range  = range(2, 10)
inertias = []
silhouettes = []
db_scores   = []
ch_scores   = []

print("🔍 Recherche du K optimal...")
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    lbl = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, lbl))
    db_scores.append(davies_bouldin_score(X_cluster, lbl))
    ch_scores.append(calinski_harabasz_score(X_cluster, lbl))
    print(f"  K={k} | Inertia={km.inertia_:.0f} | Silhouette={silhouettes[-1]:.3f} "
          f"| DBI={db_scores[-1]:.3f} | CHI={ch_scores[-1]:.0f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sélection du Nombre de Clusters K — Indices de Qualité',
             fontsize=14, fontweight='bold', color=C['navy'])

axes[0,0].plot(K_range, inertias, 'o-', color=C['blue'], lw=2)
axes[0,0].set_title('Méthode du Coude (Inertie)'); axes[0,0].set_xlabel('K')
axes[0,0].set_ylabel('Inertie intra-cluster')

axes[0,1].plot(K_range, silhouettes, 's-', color=C['green'], lw=2)
axes[0,1].set_title('Score Silhouette (↑ meilleur)'); axes[0,1].set_xlabel('K')
axes[0,1].set_ylabel('Score Silhouette')
best_k_sil = list(K_range)[np.argmax(silhouettes)]
axes[0,1].axvline(best_k_sil, color='red', ls='--',
                  label=f'Optimal K={best_k_sil}')
axes[0,1].legend()

axes[1,0].plot(K_range, db_scores, '^-', color=C['orange'], lw=2)
axes[1,0].set_title('Davies-Bouldin Index (↓ meilleur)'); axes[1,0].set_xlabel('K')
axes[1,0].set_ylabel('DBI')
best_k_db = list(K_range)[np.argmin(db_scores)]
axes[1,0].axvline(best_k_db, color='red', ls='--', label=f'Optimal K={best_k_db}')
axes[1,0].legend()

axes[1,1].plot(K_range, ch_scores, 'D-', color=C['purple'], lw=2)
axes[1,1].set_title('Calinski-Harabasz Index (↑ meilleur)'); axes[1,1].set_xlabel('K')
axes[1,1].set_ylabel('CHI')

plt.tight_layout()
plt.savefig('03_kmeans_selection.png', dpi=150, bbox_inches='tight')
plt.show()

K_OPTIMAL = best_k_sil
print(f"\n✅ K optimal retenu : {K_OPTIMAL}")

In [ ]:
# ── Clustering final avec K optimal ──────────────────────────────────
kmeans      = KMeans(n_clusters=K_OPTIMAL, random_state=SEED, n_init=30)
CLUSTER_KM  = kmeans.fit_predict(X_cluster)

df_clin['cluster_km'] = CLUSTER_KM
SC['cluster_km']      = CLUSTER_KM

# ── Profil de chaque cluster ───────────────────────────────────────────
print("\n📊 Profil clinique par cluster K-Means\n")
profile_cols = ['glycemie','IMC','CRP','HbA1c','fibers_g','H',
                'inflam','scfa','dysbiose','global']
cluster_df   = pd.concat([
    df_clin[['glycemie','IMC','CRP','HbA1c','fibers_g','diabete']],
    SC[['H','inflam','scfa','dysbiose','global']]
], axis=1)
cluster_df['cluster'] = CLUSTER_KM

profil = cluster_df.groupby('cluster').mean().round(3)
print(profil.to_string())

# ── Visualisation UMAP coloré par cluster ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'K-Means K={K_OPTIMAL} — Visualisation UMAP',
             fontsize=14, fontweight='bold', color=C['navy'])

for k in range(K_OPTIMAL):
    m = CLUSTER_KM == k
    axes[0].scatter(X_umap[m,0], X_umap[m,1], c=CLUSTER_PALETTE[k],
                    alpha=0.7, s=40, label=f'Cluster {k}',
                    edgecolors='white', lw=0.3)
axes[0].set_title('Clusters microbiomiques (UMAP)')
axes[0].legend(markerscale=1.5, fontsize=9)
axes[0].set_xlabel('UMAP1'); axes[0].set_ylabel('UMAP2')

# Boxplot score global par cluster
data_box = [SC.loc[SC['cluster_km']==k, 'global'].values for k in range(K_OPTIMAL)]
bp = axes[1].boxplot(data_box, patch_artist=True, notch=False,
                     boxprops=dict(facecolor=C['light'], color=C['navy']),
                     medianprops=dict(color=C['green'], linewidth=2.5))
for patch, col in zip(bp['boxes'], CLUSTER_PALETTE):
    patch.set_facecolor(col); patch.set_alpha(0.6)
axes[1].set_xticklabels([f'Cluster {k}' for k in range(K_OPTIMAL)])
axes[1].set_ylabel('Score Métabolique Global')
axes[1].set_title('Score Global par Cluster K-Means')
axes[1].axhline(66, color='green', ls='--', alpha=0.5, label='Seuil Optimal')
axes[1].axhline(33, color='red',   ls='--', alpha=0.5, label='Seuil Critique')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('04_kmeans_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Nommer les clusters par profil biologique ─────────────────────────
cluster_names = {}
for k in range(K_OPTIMAL):
    g   = SC.loc[SC['cluster_km']==k, 'global'].mean()
    d   = SC.loc[SC['cluster_km']==k, 'dysbiose'].mean()
    H   = SC.loc[SC['cluster_km']==k, 'H'].mean()
    diab= df_clin.loc[df_clin['cluster_km']==k, 'diabete'].mean()

    if g > 66 and H > 3:
        name = f'Cluster {k} — 🟢 Microbiome Optimal'
    elif d > 1.5 or diab > 0.3:
        name = f'Cluster {k} — 🔴 Dysbiose / Risque Élevé'
    elif g > 33:
        name = f'Cluster {k} — 🟡 Microbiome Modéré'
    else:
        name = f'Cluster {k} — 🟠 Déséquilibre Métabolique'
    cluster_names[k] = name
    print(f"{name}")
    print(f"  n={sum(CLUSTER_KM==k)} patients | Score={g:.1f} | Dysbiose={d:.2f} "
          f"| H={H:.2f} | P(diabète)={diab:.1%}\n")

---
## 8. Clustering Hiérarchique (Ward) <a id='8'></a>

In [ ]:
# ── Clustering agglomératif sur un sous-ensemble (dendrogramme) ───────
N_SUBSET    = 80
idx_subset  = np.random.choice(N_PATIENTS, N_SUBSET, replace=False)
X_sub       = X_cluster[idx_subset]

# Calcul du linkage (Ward — minimise variance intra-cluster)
Z_ward   = linkage(X_sub, method='ward', metric='euclidean')
Z_avg    = linkage(X_sub, method='average', metric='euclidean')

# Labels colorés par score global
scores_sub = SC.iloc[idx_subset]['global'].values
colors_sub = ['green' if s > 66 else 'orange' if s > 33 else 'red' for s in scores_sub]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle('Clustering Hiérarchique — Méthodes Ward & Average',
             fontsize=14, fontweight='bold', color=C['navy'])

dend_ward = dendrogram(Z_ward, ax=axes[0], leaf_rotation=90,
                        leaf_font_size=7, color_threshold=0.7*max(Z_ward[:,2]),
                        above_threshold_color=C['navy'])
axes[0].set_title('Méthode Ward')
axes[0].set_ylabel('Distance')
axes[0].axhline(0.7*max(Z_ward[:,2]), color='red', ls='--', lw=1.5,
                label=f'Seuil → K={K_OPTIMAL} clusters')
axes[0].legend()

dend_avg = dendrogram(Z_avg, ax=axes[1], leaf_rotation=90,
                       leaf_font_size=7, color_threshold=0.6*max(Z_avg[:,2]),
                       above_threshold_color=C['navy'])
axes[1].set_title('Méthode Average')
axes[1].set_ylabel('Distance')

plt.tight_layout()
plt.savefig('05_hierarchical.png', dpi=150, bbox_inches='tight')
plt.show()

# Clustering hiérarchique sur toute la cohorte
hier_model   = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage='ward')
CLUSTER_HIER = hier_model.fit_predict(X_cluster)
SC['cluster_hier'] = CLUSTER_HIER

# Accord entre K-Means et Hiérarchique
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(CLUSTER_KM, CLUSTER_HIER)
print(f"\n✅ Adjusted Rand Index (K-Means vs Ward) : {ari:.3f}")
print(f"   (1.0 = accord parfait · 0.0 = aléatoire)")

---
## 9. Modèles de Mélange Gaussien (GMM) <a id='9'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GMM — Probabilités d'appartenance à chaque profil microbiomique
# Avantage sur K-Means : un patient peut APPARTENIR à PLUSIEURS profils
# avec des probabilités → utile pour les cas intermédiaires
# ══════════════════════════════════════════════════════════════════════

print("🔮 Modèles de Mélange Gaussien (GMM)...")

# Sélection du nombre de composantes via BIC et AIC
bics, aics = [], []
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                           random_state=SEED, n_init=5)
    gmm.fit(X_cluster)
    bics.append(gmm.bic(X_cluster))
    aics.append(gmm.aic(X_cluster))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(2,9), bics, 'o-', color=C['blue'],   lw=2, label='BIC (↓ meilleur)')
ax.plot(range(2,9), aics, 's-', color=C['orange'], lw=2, label='AIC (↓ meilleur)')
k_best_bic = list(range(2,9))[np.argmin(bics)]
k_best_aic = list(range(2,9))[np.argmin(aics)]
ax.axvline(k_best_bic, color=C['blue'],   ls='--', label=f'BIC opt. K={k_best_bic}')
ax.axvline(k_best_aic, color=C['orange'], ls='--', label=f'AIC opt. K={k_best_aic}')
ax.set_xlabel('Nombre de composantes K')
ax.set_ylabel('Critère')
ax.set_title('Sélection GMM via BIC/AIC',
             fontweight='bold', color=C['navy'])
ax.legend()
plt.tight_layout()
plt.savefig('06_gmm_bic.png', dpi=150, bbox_inches='tight')
plt.show()

K_GMM = k_best_bic
gmm_final = GaussianMixture(n_components=K_GMM, covariance_type='full',
                              random_state=SEED, n_init=10)
gmm_final.fit(X_cluster)

CLUSTER_GMM     = gmm_final.predict(X_cluster)
PROBA_GMM       = gmm_final.predict_proba(X_cluster)  # Probabilités d'appartenance
SC['cluster_gmm']   = CLUSTER_GMM
for k in range(K_GMM):
    SC[f'proba_gmm_{k}'] = PROBA_GMM[:, k]

print(f"\n✅ GMM entraîné avec K={K_GMM} composantes")
print("\n📊 Probabilités d'appartenance (5 premiers patients) :")
proba_cols = [f'proba_gmm_{k}' for k in range(K_GMM)]
print(SC[proba_cols].head().round(3).to_string())

# Visualisation UMAP + incertitude GMM
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('GMM — Clusters et Incertitude d\'Appartenance',
             fontsize=14, fontweight='bold', color=C['navy'])

for k in range(K_GMM):
    m = CLUSTER_GMM == k
    axes[0].scatter(X_umap[m,0], X_umap[m,1], c=CLUSTER_PALETTE[k],
                    alpha=0.7, s=40, label=f'GMM {k}',
                    edgecolors='white', lw=0.3)
axes[0].set_title('Clusters GMM (UMAP)')
axes[0].legend(fontsize=9)

# Entropie = incertitude d'appartenance
entropy = -np.sum(PROBA_GMM * np.log(PROBA_GMM + 1e-10), axis=1)
sc = axes[1].scatter(X_umap[:,0], X_umap[:,1], c=entropy, cmap='plasma',
                      alpha=0.7, s=40)
plt.colorbar(sc, ax=axes[1], label='Entropie (incertitude)')
axes[1].set_title('Incertitude d\'Appartenance (Entropie GMM)\nRouge = cas ambigus (frontière entre profils)')

plt.tight_layout()
plt.savefig('07_gmm_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. DBSCAN — Détection d'Anomalies Microbiomiques <a id='10'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DBSCAN — Détecte les profils microbiomiques atypiques (outliers)
# Label -1 = ANOMALIE (patient avec profil microbiomique unique)
# Usage clinique : alerter le médecin sur des cas inhabituels
# ══════════════════════════════════════════════════════════════════════

# Trouver epsilon optimal via k-distance graph
knn   = NearestNeighbors(n_neighbors=5)
knn.fit(X_cluster)
dist, _ = knn.kneighbors(X_cluster)
k_dist  = np.sort(dist[:, 4])[::-1]  # 5ème plus proche voisin

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_dist, color=C['purple'], lw=2)
ax.axhline(np.percentile(k_dist, 10), color='red', ls='--',
           label=f"ε = {np.percentile(k_dist, 10):.3f} (P10)")
ax.set_xlabel('Patients (triés par distance)')
ax.set_ylabel('Distance au 5ème voisin')
ax.set_title('K-Distance Graph — Sélection de ε pour DBSCAN',
             fontweight='bold', color=C['navy'])
ax.legend()
plt.tight_layout()
plt.savefig('08_dbscan_epsilon.png', dpi=150, bbox_inches='tight')
plt.show()

EPS = np.percentile(k_dist, 10)
dbscan     = DBSCAN(eps=EPS, min_samples=5)
CLUSTER_DB = dbscan.fit_predict(X_cluster)
SC['cluster_dbscan']  = CLUSTER_DB
SC['anomalie']        = (CLUSTER_DB == -1).astype(int)
df_clin['anomalie']   = (CLUSTER_DB == -1).astype(int)

n_outliers = (CLUSTER_DB == -1).sum()
n_clusters = len(set(CLUSTER_DB)) - (1 if -1 in CLUSTER_DB else 0)

print(f"✅ DBSCAN : {n_clusters} clusters détectés · {n_outliers} anomalies ({n_outliers/N_PATIENTS:.1%})")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('DBSCAN — Détection d\'Anomalies Microbiomiques',
             fontsize=14, fontweight='bold', color=C['navy'])

# UMAP avec anomalies
mask_core = CLUSTER_DB >= 0
mask_out  = CLUSTER_DB == -1
axes[0].scatter(X_umap[mask_core,0], X_umap[mask_core,1],
                c=[CLUSTER_PALETTE[c % len(CLUSTER_PALETTE)] for c in CLUSTER_DB[mask_core]],
                alpha=0.6, s=30, label='Patients classés')
axes[0].scatter(X_umap[mask_out,0], X_umap[mask_out,1],
                c='black', marker='x', s=80, lw=2,
                label=f'Anomalies (n={n_outliers})')
axes[0].set_title('Clusters DBSCAN + Anomalies (×)')
axes[0].legend()

# Profil des anomalies vs population normale
anomalies_scores = SC[SC['anomalie']==1][['H','inflam','scfa','dysbiose','global']].mean()
normal_scores    = SC[SC['anomalie']==0][['H','inflam','scfa','dysbiose','global']].mean()
comparison       = pd.DataFrame({'Anomalies': anomalies_scores, 'Normaux': normal_scores})

comparison.plot(kind='bar', ax=axes[1], color=[C['red'], C['green']], edgecolor='white')
axes[1].set_title('Profil Scores — Anomalies vs Population Normale')
axes[1].set_ylabel('Score moyen')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig('09_dbscan_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Patients anomalies — leurs scores biologiques :")
print(comparison.round(3).to_string())

---
## 11. Analyse de Co-abondance (Réseau Microbien) <a id='11'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CO-ABONDANCE — Quelles bactéries coexistent ensemble ?
# Corrélation de Spearman sur abondances relatives
# Usage : identifier les modules fonctionnels microbiens
# ══════════════════════════════════════════════════════════════════════

KEY_BACTERIA = (PATHOBIONTES + BUTYRATE_PRODUCERS + BENEFICIAL)[:25]
KEY_BACTERIA = [b for b in KEY_BACTERIA if b in X_rel.columns][:20]

X_key    = X_rel[KEY_BACTERIA]
corr_mat, p_mat = spearmanr(X_key)

if isinstance(corr_mat, float):
    corr_mat = np.eye(len(KEY_BACTERIA))
    p_mat    = np.ones((len(KEY_BACTERIA), len(KEY_BACTERIA)))

corr_df = pd.DataFrame(corr_mat, index=KEY_BACTERIA, columns=KEY_BACTERIA)
p_df    = pd.DataFrame(p_mat,    index=KEY_BACTERIA, columns=KEY_BACTERIA)

# Masque : corrélations non-significatives (p > 0.05)
sig_mask = p_df > 0.05
corr_sig = corr_df.copy()
corr_sig[sig_mask] = 0

# Annotation des types
short_names = [b.replace('_',' ')[:20] for b in KEY_BACTERIA]
colors_rows  = ['#CC0000' if b in PATHOBIONTES
                else '#009944' if b in BUTYRATE_PRODUCERS
                else '#0066CC' for b in KEY_BACTERIA]

fig, ax = plt.subplots(figsize=(13, 11))

mask_tri = np.zeros_like(corr_sig, dtype=bool)
mask_tri[np.triu_indices_from(mask_tri)] = True

sns.heatmap(corr_sig, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, mask=mask_tri,
            xticklabels=short_names, yticklabels=short_names,
            cbar_kws={'label': 'Corrélation Spearman (p<0.05)'})

ax.set_title('Réseau de Co-abondance Microbienne\n(Corrélations de Spearman significatives)',
             fontsize=13, fontweight='bold', color=C['navy'])

# Colorier les labels selon le type de bactérie
for label, color in zip(ax.get_xticklabels(), colors_rows):
    label.set_color(color)
for label, color in zip(ax.get_yticklabels(), colors_rows):
    label.set_color(color)

patches = [
    mpatches.Patch(color='#CC0000', label='Pathobionte'),
    mpatches.Patch(color='#009944', label='Producteur SCFA'),
    mpatches.Patch(color='#0066CC', label='Bénéfique autre'),
]
ax.legend(handles=patches, loc='upper right', fontsize=9)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('10_coabondance.png', dpi=150, bbox_inches='tight')
plt.show()

# Top corrélations positives et négatives
corr_pairs = []
for i in range(len(KEY_BACTERIA)):
    for j in range(i+1, len(KEY_BACTERIA)):
        if p_df.iloc[i,j] < 0.05:
            corr_pairs.append({
                'Bactérie A': KEY_BACTERIA[i],
                'Bactérie B': KEY_BACTERIA[j],
                'r_Spearman': round(corr_df.iloc[i,j], 3),
                'p_value'   : round(p_df.iloc[i,j], 4),
                'Type'      : 'Mutualiste' if corr_df.iloc[i,j]>0 else 'Compétiteur',
            })

df_pairs = pd.DataFrame(corr_pairs).sort_values('r_Spearman', key=abs, ascending=False)
print(f"\n✅ {len(df_pairs)} corrélations significatives (p<0.05)")
print("\nTop 10 associations microbiennes :")
print(df_pairs.head(10).to_string(index=False))

---
## PARTIE D — APPRENTISSAGE SUPERVISÉ

---
## 12. Modèles Supervisés <a id='12'></a>

In [ ]:
# Feature matrix enrichie : PCA + Scores + Clinique + Cluster (non-supervisé comme feature)
X_feat = np.hstack([
    X_pca_90,
    SC[['H','inflam','scfa','dysbiose','global']].values,
    X_clin_sc.values,
    CLUSTER_KM.reshape(-1, 1),
    PROBA_GMM,
])

feat_names = (
    [f'PC{i+1}' for i in range(n_comp90)] +
    ['H','inflam','scfa','dysbiose','global'] +
    list(X_clin_sc.columns) +
    ['cluster_km'] +
    [f'proba_gmm_{k}' for k in range(K_GMM)]
)

y = df_clin['diabete'].values
X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"✅ Features enrichies (supervisé + non-supervisé) : {X_feat.shape[1]}")

# Entraînement des 3 modèles
models = {
    'Logistique'  : Pipeline([('sc',StandardScaler()),
                               ('clf',LogisticRegression(C=1.0,max_iter=1000,
                                                          class_weight='balanced',
                                                          random_state=SEED))]),
    'RandomForest': RandomForestClassifier(n_estimators=500, max_depth=10,
                                            class_weight='balanced',
                                            random_state=SEED, n_jobs=-1),
    'XGBoost'     : xgb.XGBClassifier(n_estimators=500, max_depth=6,
                                       learning_rate=0.05, subsample=0.8,
                                       colsample_bytree=0.8,
                                       scale_pos_weight=((y_train==0).sum()/(y_train==1).sum()),
                                       eval_metric='logloss', random_state=SEED, n_jobs=-1),
}

print("\n🤖 Entraînement des modèles supervisés...")
trained, metrics = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    auc     = roc_auc_score(y_test, y_proba)
    f1      = f1_score(y_test, y_pred, average='weighted')
    trained[name]  = model
    metrics[name]  = {'AUC':auc,'F1':f1,'proba':y_proba}
    print(f"  {name:<15} AUC={auc:.3f}  F1={f1:.3f}")

# Courbes ROC
fig, ax = plt.subplots(figsize=(9, 7))
colors  = [C['blue'], C['green'], C['orange']]
for (name, m), col in zip(metrics.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, m['proba'])
    ax.plot(fpr, tpr, color=col, lw=2.5, label=f"{name} (AUC={m['AUC']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1.5,alpha=0.5,label='Aléatoire')
ax.set_xlabel('Taux Faux Positifs'); ax.set_ylabel('Taux Vrais Positifs')
ax.set_title('Courbes ROC — Prédiction Risque Diabète T2',
             fontsize=14, fontweight='bold', color=C['navy'])
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('11_roc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. SHAP & Métriques de Validation <a id='13'></a>

In [ ]:
# SHAP sur XGBoost
print("🔍 Calcul SHAP values...")
explainer   = shap.TreeExplainer(trained['XGBoost'])
shap_values = explainer.shap_values(X_test[:150])

fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test[:150],
                  feature_names=feat_names[:X_test.shape[1]],
                  max_display=20, show=False)
plt.title('SHAP — Importance & Direction des Features (XGBoost)',
          fontsize=13, fontweight='bold', color=C['navy'])
plt.tight_layout()
plt.savefig('12_shap.png', dpi=150, bbox_inches='tight')
plt.show()

# K-Fold validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("\n🔁 Cross-Validation K-Fold (k=5) :")
for name, model in trained.items():
    cv = cross_val_score(model, X_feat, y, cv=kfold, scoring='roc_auc', n_jobs=-1)
    print(f"  {name:<15} AUC = {cv.mean():.3f} ± {cv.std():.3f}")

---
## PARTIE E — MOTEUR DE RECOMMANDATIONS

---
## 14. Recommandations basées sur les clusters <a id='14'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RECOMMANDATIONS BASÉES SUR LE CLUSTERING (Non Supervisé)
#
# Principe : identifier les caractéristiques distinctives de chaque
# cluster et formuler des recommandations spécifiques à ce profil.
# ══════════════════════════════════════════════════════════════════════

# Base de connaissances — recommandations par profil cluster
CLUSTER_RECS = {
    'optimal': {
        'label'       : '🟢 Profil Optimal',
        'description' : 'Microbiome diversifié, faible dysbiose, bonne production SCFA.',
        'alimentation': [
            "✅ Continuer votre alimentation actuelle — riche en fibres locales",
            "🫘 Maintenir la consommation de niébé (2–3×/semaine) — fibres fermentescibles",
            "🌾 Varier les céréales : mil, sorgho, fonio, riz complet en alternance",
            "🥬 5 portions de légumes/jour minimum — diversité des polyphénols",
        ],
        'probiotiques' : ["✅ Aucune supplémentation urgente — microbiome en bon état",
                          "🥛 Maintenir la consommation de lait caillé fermenté local"],
        'lifestyle'    : ["😴 7–8h de sommeil — maintien de la diversité microbienne",
                          "🏃 30 min d'activité physique × 5j — microbiome athlétique",
                          "🌿 Jardinage ou contact sol — exposition aux microbes environnementaux"],
        'suivi'        : "Prochain kit dans 12 mois ou si symptômes digestifs apparaissent.",
        'alerte'       : None,
    },
    'modere': {
        'label'       : '🟡 Profil Modéré',
        'description' : 'Microbiome en déséquilibre partiel. Quelques carences identifiées.',
        'alimentation': [
            "🔼 Augmenter les fibres : cible 30g/jour (vous êtes probablement sous les 20g)",
            "🫘 Niébé, lentilles, pois chiches — fibres solubles et SCFA précurseurs",
            "🐟 Poisson 3×/semaine — oméga-3 anti-inflammatoires (tilapia, sardine, capitaine)",
            "🍃 Feuilles de baobab séchées (1 cuillère/jour) — prébiotique africain puissant",
            "🚫 Réduire de 30% les aliments ultra-transformés et boissons sucrées",
        ],
        'probiotiques' : ["Lactobacillus rhamnosus GG — 10⁹ UFC × 30 jours",
                          "Yaourt fermenté local quotidien (lait de vache ou chèvre)"],
        'lifestyle'    : ["😴 Optimiser le sommeil — se coucher avant 23h",
                          "🚶 Marche rapide 30 min/jour — minimum 5 jours/semaine",
                          "💧 2L d'eau/jour — hydratation essentielle au transit",
                          "🧘 Technique de cohérence cardiaque (5 min × 3/jour)"],
        'suivi'        : "Prochain kit dans 6 mois pour évaluer la progression.",
        'alerte'       : None,
    },
    'risque': {
        'label'       : '🔴 Profil à Risque / Dysbiose',
        'description' : 'Dysbiose significative, faible diversité, pathobiontes élevés. Intervention urgente.',
        'alimentation': [
            "🚨 PRIORITÉ : Éliminer totalement sucres raffinés et farine blanche — 30 jours",
            "🫘 Niébé et légumineuses quotidiennement — reconstruction de la flore bénéfique",
            "🌾 Céréales uniquement complètes : mil complet, sorgho, épeautre",
            "🐟 Poisson gras × 4/semaine + huile de poisson si glycémie élevée",
            "🫙 Légumes lacto-fermentés quotidiens — carottes, concombre, chou africain",
            "🧄 Ail + oignon crus quotidiennement — FOS prébiotiques + allicine anti-bactérienne",
            "☕ Thé vert (2 tasses/jour) — polyphénols restructurant le microbiome",
        ],
        'probiotiques' : [
            "⚠️ Protocole intensif 90 jours — consultation nutritionniste recommandée",
            "Saccharomyces boulardii (500mg × 2/jour) — protection barrière intestinale",
            "Multi-souches : Lactobacillus + Bifidobacterium + Akkermansia — formule complète",
            "Prébiotiques : Inuline-FOS 5g/jour — nourrir les bactéries bénéfiques",
        ],
        'lifestyle'    : [
            "🛑 Arrêt total alcool et tabac — effet délétère direct sur le microbiome",
            "😴 Sommeil : MINIMUM 8h — le microbiome se répare pendant le sommeil profond",
            "🏊 Exercice modéré (pas intense) : natation, marche — éviter le surentraînement",
            "💊 Si antibiotiques prescrits : demander un prébiotique concomitant au médecin",
            "🧘 Réduire le stress chronique — cortisol élevé détruit Faecalibacterium prausnitzii",
        ],
        'suivi'        : "Prochain kit OBLIGATOIRE dans 3 mois. Consultation médicale recommandée.",
        'alerte'       : "⚠️ Dysbiose sévère détectée. Consulter un gastro-entérologue ou médecin interniste.",
    },
    'anomalie': {
        'label'       : '🔵 Profil Atypique (DBSCAN)',
        'description' : 'Profil microbiomique unique — hors des patterns habituels. Investigation approfondie.',
        'alimentation': [
            "🔬 Profil inhabituel — recommandations génériques appliquées en attente d'analyse approfondie",
            "🫘 Fibres fermentescibles quotidiennes — base sûre pour tout profil",
            "🌈 Diversité alimentaire maximale — minimum 30 espèces végétales/semaine",
        ],
        'probiotiques' : ["⚠️ Éviter les probiotiques sans avis médical — profil atypique"],
        'lifestyle'    : ["📋 Consultation spécialisée recommandée (gastro-entérologue)"],
        'suivi'        : "Prochain kit dans 2 mois avec analyse métagénomique complète (shotgun).",
        'alerte'       : "🔵 Profil microbiomique hors norme. Analyse métagénomique shotgun recommandée.",
    }
}

def get_cluster_profile(cluster_km, score_global, dysbiose, is_anomaly):
    """Détermine le profil de recommandation basé sur le clustering non supervisé."""
    if is_anomaly:
        return 'anomalie'
    if score_global > 66 and dysbiose < 1.0:
        return 'optimal'
    if dysbiose > 1.5 or score_global < 33:
        return 'risque'
    return 'modere'

print("✅ Base de recommandations par cluster définie")
print(f"   Profils : optimal · modéré · risque · anomalie")

# Appliquer à toute la cohorte
SC['profil_cluster'] = [
    get_cluster_profile(
        row['cluster_km'], row['global'], row['dysbiose'], row['anomalie']
    )
    for _, row in SC.iterrows()
]

dist_profils = SC['profil_cluster'].value_counts()
print("\n📊 Distribution des profils dans la cohorte :")
for profil, count in dist_profils.items():
    pct = count / N_PATIENTS * 100
    bar = '█' * int(pct / 2)
    print(f"  {profil:<12} : {count:3d} patients ({pct:.1f}%)  {bar}")

---
## 15. Filtrage Collaboratif — Patients Similaires <a id='15'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FILTRAGE COLLABORATIF
#
# Principe : "Les patients qui vous ressemblent microbiomiquement
# ont amélioré leur santé en faisant X → nous vous recommandons X"
#
# Utilise KNN dans l'espace de l'Autoencoder (représentation compacte)
# ══════════════════════════════════════════════════════════════════════

class CollaborativeRecommender:
    """
    Système de recommandation basé sur la similarité entre patients.

    Étapes :
    1. Trouver les K patients les plus similaires (KNN dans espace latent)
    2. Extraire leurs comportements alimentaires et résultats cliniques
    3. Recommander ce qui a fonctionné pour leurs "jumeaux microbiomiques"
    """

    def __init__(self, Z_latent, df_clin, X_rel, k=10):
        self.Z       = Z_latent
        self.clin    = df_clin
        self.rel     = X_rel
        self.k       = k
        self.knn     = NearestNeighbors(n_neighbors=k+1, metric='euclidean')
        self.knn.fit(Z_latent)
        self.patients= list(df_clin.index)

    def find_similar(self, patient_idx):
        """Retourne les K patients les plus similaires."""
        z_query  = self.Z[patient_idx].reshape(1, -1)
        dists, idxs = self.knn.kneighbors(z_query)
        similar_idxs = idxs[0][1:]  # Exclure lui-même
        similar_dists= dists[0][1:]
        return similar_idxs, similar_dists

    def collaborative_recs(self, patient_idx, SC_df):
        """Génère des recommandations basées sur les patients similaires sains."""
        sim_idxs, sim_dists = self.find_similar(patient_idx)

        # Filtrer : parmi les voisins, ceux qui ont un MEILLEUR score global
        target_score = SC_df.iloc[patient_idx]['global']
        healthy_idxs = [i for i in sim_idxs
                        if SC_df.iloc[i]['global'] > target_score + 10]

        if len(healthy_idxs) < 2:
            healthy_idxs = sim_idxs[:5]  # Fallback : 5 plus proches

        # Profil alimentaire des "jumeaux sains"
        healthy_diet = self.clin.iloc[healthy_idxs][
            ['fibers_g','sugar_g','millet_freq','niebe_freq','fish_portions']
        ].mean()

        # Comparaison avec le patient cible
        patient_diet = self.clin.iloc[patient_idx][
            ['fibers_g','sugar_g','millet_freq','niebe_freq','fish_portions']
        ]

        recs = []
        recs.append(f"👥 Basé sur {len(healthy_idxs)} patients similaires avec meilleur score microbiome :")

        if healthy_diet['fibers_g'] > patient_diet['fibers_g'] + 5:
            diff = healthy_diet['fibers_g'] - patient_diet['fibers_g']
            recs.append(f"  🌾 Vos jumeaux microbiomiques consomment {diff:.0f}g/j de fibres de plus → augmenter")

        if healthy_diet['sugar_g'] < patient_diet['sugar_g'] - 10:
            diff = patient_diet['sugar_g'] - healthy_diet['sugar_g']
            recs.append(f"  🍬 Réduire le sucre de {diff:.0f}g/j — vos jumeaux sains consomment moins")

        if healthy_diet['niebe_freq'] > patient_diet['niebe_freq'] + 1:
            recs.append(f"  🫘 Augmenter le niébé : vos pairs sains en mangent {healthy_diet['niebe_freq']:.1f}×/sem")

        if healthy_diet['fish_portions'] > patient_diet['fish_portions'] + 1:
            recs.append(f"  🐟 Plus de poisson : vos jumeaux sains : {healthy_diet['fish_portions']:.1f} portions/sem")

        if healthy_diet['millet_freq'] > patient_diet['millet_freq']:
            recs.append(f"  🌾 Mil/sorgho plus fréquent ({healthy_diet['millet_freq']:.1f}× vs {patient_diet['millet_freq']:.1f}×/sem)")

        if len(recs) == 1:
            recs.append("  ✅ Votre alimentation est similaire à celle de vos jumeaux microbiomiques sains")

        # Score de confiance (basé sur la similarité)
        confidence = max(0, 1 - np.mean(sim_dists[:len(healthy_idxs)]) / 10)
        recs.append(f"  📊 Confiance collaborative : {confidence:.0%}")

        return {
            'recs'         : recs,
            'n_similar'    : len(healthy_idxs),
            'confidence'   : confidence,
            'healthy_profile': healthy_diet.round(2).to_dict(),
            'patient_profile': patient_diet.round(2).to_dict(),
        }


collab = CollaborativeRecommender(Z_latent, df_clin, X_rel, k=20)
print("✅ Système de filtrage collaboratif initialisé (KNN k=20 dans espace latent autoencoder)")

# Test sur Patient P0042
test_idx = 42
result   = collab.collaborative_recs(test_idx, SC)
print(f"\n📋 Recommandations collaboratives — P{test_idx:04d}")
print("─" * 50)
for r in result['recs']:
    print(r)

print("\n  Profil alimentaire voisins sains vs patient :")
comp = pd.DataFrame({'Voisins sains': result['healthy_profile'],
                      'Patient cible': result['patient_profile']})
print(comp.to_string())

---
## 16. Système de Règles Cliniques Expertes <a id='16'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SYSTÈME DE RÈGLES EXPERTES
#
# Règles IF-THEN basées sur la littérature scientifique
# + calibration sur les clusters identifiés
# Utilisé comme couche d'interprétabilité sur les prédictions ML
# ══════════════════════════════════════════════════════════════════════

class ExpertRuleEngine:
    """
    Moteur de règles cliniques expertes.

    Chaque règle produit :
        - Un message de recommandation
        - Une priorité (1=urgent · 2=important · 3=conseil)
        - Une évidence scientifique (référence)
    """

    RULES = [
        # ── RÈGLES DYSBIOSE ──────────────────────────────────────────
        {
            'id'       : 'R001',
            'condition': lambda s: s['dysbiose'] > 2.0,
            'priority' : 1,
            'category' : '🚨 Alerte Clinique',
            'message'  : "Dysbiose sévère (D>{dysbiose:.2f}) — Référer à gastro-entérologue",
            'evidence' : "Sonnenburg & Bäckhed, Nature 2016",
            'action'   : "Protocole probiotiques intensif 90j + alimentation reconstructrice",
        },
        {
            'id'       : 'R002',
            'condition': lambda s: 1.5 < s['dysbiose'] <= 2.0,
            'priority' : 2,
            'category' : '⚠️ Déséquilibre Microbien',
            'message'  : "Dysbiose modérée (D={dysbiose:.2f}) — Intervention nutritionnelle recommandée",
            'evidence' : "Clemente et al., Science 2012",
            'action'   : "Augmenter fibres fermentescibles (niébé, mil, baobab) + probiotiques",
        },
        # ── RÈGLES SHANNON / DIVERSITÉ ────────────────────────────────
        {
            'id'       : 'R010',
            'condition': lambda s: s['H'] < 1.5,
            'priority' : 1,
            'category' : '🚨 Diversité Critique',
            'message'  : "Indice Shannon H={H:.2f} — diversité microbienne extrêmement faible",
            'evidence' : "Turnbaugh et al., Nature 2006",
            'action'   : "Diversification alimentaire urgente — 30 espèces végétales/semaine",
        },
        {
            'id'       : 'R011',
            'condition': lambda s: 1.5 <= s['H'] < 2.5,
            'priority' : 2,
            'category' : '⚠️ Faible Diversité',
            'message'  : "H={H:.2f} — diversité microbienne insuffisante",
            'evidence' : "Yatsunenko et al., Nature 2012",
            'action'   : "Diversifier : légumes variés, fruits locaux, céréales alternatives",
        },
        # ── RÈGLES INFLAMMATION ───────────────────────────────────────
        {
            'id'       : 'R020',
            'condition': lambda s: s['inflam'] > 1.5,
            'priority' : 1,
            'category' : '🚨 Inflammation Élevée',
            'message'  : "Score inflammatoire élevé (S={inflam:.2f}) — action anti-inflammatoire urgente",
            'evidence' : "Sonnenburg et al., Cell 2021",
            'action'   : "Oméga-3 quotidiens (poisson gras), polyphénols (thé, tamarin), réduire sucres",
        },
        {
            'id'       : 'R021',
            'condition': lambda s: 0.5 < s['inflam'] <= 1.5,
            'priority' : 2,
            'category' : '⚠️ Inflammation Modérée',
            'message'  : "Score inflammatoire modéré (S={inflam:.2f})",
            'evidence' : "Makki et al., Cell Host Microbe 2018",
            'action'   : "Augmenter légumes colorés, réduire aliments transformés",
        },
        # ── RÈGLES SCFA ───────────────────────────────────────────────
        {
            'id'       : 'R030',
            'condition': lambda s: s['scfa'] < 0.01,
            'priority' : 2,
            'category' : '⚠️ Déficit SCFA',
            'message'  : "Production SCFA très faible (S={scfa:.4f}) — butyrate insuffisant",
            'evidence' : "Flint et al., Nat Rev Microbiol 2012",
            'action'   : "Fibres résistantes : banane verte, igname cuit-refroidi, son de mil",
        },
        # ── RÈGLES CLINIQUES CROISÉES ─────────────────────────────────
        {
            'id'       : 'R040',
            'condition': lambda s: s.get('glycemie',5) > 7.0,
            'priority' : 1,
            'category' : '🚨 Alerte Glycémique',
            'message'  : "Glycémie={glycemie:.1f} mmol/L > 7.0 — consultation médicale urgente",
            'evidence' : "IFD Guidelines 2023",
            'action'   : "URGENCE : Consultation endocrinologue + régime à faible index glycémique",
        },
        {
            'id'       : 'R041',
            'condition': lambda s: 5.6 < s.get('glycemie',5) <= 7.0,
            'priority' : 2,
            'category' : '⚠️ Prédiabète',
            'message'  : "Glycémie en zone prédiabétique ({glycemie:.1f} mmol/L)",
            'evidence' : "WHO Diabetes Report 2023",
            'action'   : "Réduire sucres simples, augmenter fibres, exercice 150 min/semaine",
        },
        {
            'id'       : 'R050',
            'condition': lambda s: s.get('IMC',25) > 30,
            'priority' : 2,
            'category' : '⚠️ Obésité',
            'message'  : "IMC={IMC:.1f} — obésité et risque métabolique élevé",
            'evidence' : "Turnbaugh et al., Nature 2009",
            'action'   : "Restructuration microbiome via régime méditerranéen africain + exercice",
        },
        {
            'id'       : 'R060',
            'condition': lambda s: s.get('antibio_3mois',0) == 1,
            'priority' : 2,
            'category' : '⚠️ Post-Antibiotiques',
            'message'  : "Antibiotiques récents (< 3 mois) — flore perturbée",
            'evidence' : "Palleja et al., Nat Microbiol 2018",
            'action'   : "Probiotiques multi-souches × 60 jours + prébiotiques pour reconstruction",
        },
        # ── RÈGLES POSITIVES (Renforcement) ───────────────────────────
        {
            'id'       : 'R070',
            'condition': lambda s: s['H'] > 3.5 and s['dysbiose'] < 0.8,
            'priority' : 3,
            'category' : '✅ Excellent Profil',
            'message'  : "Microbiome exceptionnel (H={H:.2f}, D={dysbiose:.2f}) — maintenir les habitudes",
            'evidence' : "Sonnenburg Lab 2022",
            'action'   : "Maintien : variété alimentaire, fermentés, activité physique régulière",
        },
        {
            'id'       : 'R071',
            'condition': lambda s: s['scfa'] > 0.10,
            'priority' : 3,
            'category' : '✅ Production SCFA Optimale',
            'message'  : "Bonne production de butyrate — santé colique et métabolique protégée",
            'evidence' : "Canani et al., Nat Rev Gastro 2011",
            'action'   : "Continuer consommation régulière de légumineuses et fibres résistantes",
        },
    ]

    def apply(self, scores_dict):
        """
        Applique toutes les règles et retourne les alertes déclenchées,
        triées par priorité.
        """
        triggered = []
        for rule in self.RULES:
            try:
                if rule['condition'](scores_dict):
                    msg = rule['message'].format(**scores_dict)
                    triggered.append({
                        'id'      : rule['id'],
                        'priority': rule['priority'],
                        'category': rule['category'],
                        'message' : msg,
                        'action'  : rule['action'],
                        'evidence': rule['evidence'],
                    })
            except (KeyError, TypeError):
                pass  # Données manquantes pour cette règle

        return sorted(triggered, key=lambda x: x['priority'])


expert = ExpertRuleEngine()

# Test sur Patient P0042
p_scores = SC.iloc[42].to_dict()
p_scores.update(df_clin.iloc[42].to_dict())

triggered_rules = expert.apply(p_scores)
print(f"\n📋 Règles déclenchées — P{42:04d}")
print("═" * 60)
for r in triggered_rules:
    print(f"\n  [{r['id']}] P{r['priority']} {r['category']}")
    print(f"  {r['message']}")
    print(f"  → {r['action']}")
    print(f"  📚 {r['evidence']}")

print(f"\n✅ {len(triggered_rules)} règles déclenchées")

---
## 17. Moteur de Recommandations Unifié BiomeX <a id='17'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MOTEUR UNIFIÉ — Combine les 3 sources de recommandations :
#
# 1. Clustering non supervisé  → profil microbiomique global
# 2. Filtrage collaboratif    → patients similaires
# 3. Règles expertes          → alertes cliniques
# 4. Prédictions supervisées  → risques quantifiés (ML)
#
# Output : rapport de recommandations structuré et priorisé
# ══════════════════════════════════════════════════════════════════════

class BiomeXRecommendationEngine:
    """
    Moteur central de recommandations BiomeX.

    Architecture à 4 couches :
        L1 — Scores biologiques (Shannon, SCFA, Dysbiose, Global)
        L2 — Clustering non-supervisé (K-Means + GMM + DBSCAN)
        L3 — Filtrage collaboratif (KNN Autoencoder)
        L4 — Règles cliniques expertes + Prédictions ML
    """

    def __init__(self, cluster_recs, collab_rec, expert_engine,
                 ml_models, pca, n_pca):
        self.cluster_recs = cluster_recs
        self.collab       = collab_rec
        self.expert       = expert_engine
        self.ml           = ml_models
        self.pca          = pca
        self.n_pca        = n_pca

    def generate(self, patient_idx, SC_df, df_clin_full,
                 X_feat_full, feat_names_list):
        """
        Génère le rapport de recommandations complet pour un patient.
        """
        pid = SC_df.index[patient_idx]
        row = SC_df.iloc[patient_idx]
        clin= df_clin_full.iloc[patient_idx]

        scores = row.to_dict()
        scores.update(clin.to_dict())

        # ── L1 : Profil cluster ───────────────────────────────────────
        profil_key  = row.get('profil_cluster', 'modere')
        profil_data = self.cluster_recs.get(profil_key,
                                             self.cluster_recs['modere'])

        # ── L2 : Prédictions ML (risques quantifiés) ──────────────────
        x = X_feat_full[patient_idx:patient_idx+1]
        ml_probas = {
            name: float(model.predict_proba(x)[0, 1])
            for name, model in self.ml.items()
        }
        ensemble_proba = (
            0.4 * ml_probas['XGBoost'] +
            0.4 * ml_probas['RandomForest'] +
            0.2 * ml_probas['Logistique']
        )

        def risk_label(p):
            if p < 0.25:  return '🟢 Faible'
            if p < 0.60:  return '🟡 Modéré'
            return '🔴 Élevé'

        # ── L3 : Filtrage collaboratif ────────────────────────────────
        collab_result = self.collab.collaborative_recs(patient_idx, SC_df)

        # ── L4 : Règles expertes ──────────────────────────────────────
        expert_rules = self.expert.apply(scores)

        # ── Assemblage du rapport final ───────────────────────────────
        return {
            'patient_id' : pid,
            'date'       : '2026-02-01',

            'SECTION_1_SCORES': {
                'Shannon H (Diversité)'   : f"{row['H']:.2f}  {'✅' if row['H']>3 else '⚠️' if row['H']>2 else '🔴'}",
                'Inflammation'            : f"{row['inflam']:.2f}  {'🔴 Élevée' if row['inflam']>1 else '⚠️ Modérée' if row['inflam']>0 else '✅ Faible'}",
                'SCFA / Butyrate'         : f"{row['scfa']:.4f}  {'✅' if row['scfa']>0.05 else '⚠️ Insuffisant'}",
                'Dysbiose D'              : f"{row['dysbiose']:.2f}  {'✅ <1.0' if row['dysbiose']<1 else '⚠️ Mod.' if row['dysbiose']<1.5 else '🔴 >1.5'}",
                'Score Global'            : f"{row['global']:.1f}/100",
                'Profil Cluster'          : profil_data['label'],
                'Anomalie DBSCAN'         : '⚠️ Profil atypique' if row.get('anomalie',0)==1 else '✅ Normal',
            },

            'SECTION_2_RISQUES_ML': {
                'Risque Diabète T2 (LogReg)' : f"{ml_probas['Logistique']:.1%}  {risk_label(ml_probas['Logistique'])}",
                'Risque Diabète T2 (RF)'     : f"{ml_probas['RandomForest']:.1%}  {risk_label(ml_probas['RandomForest'])}",
                'Risque Diabète T2 (XGB)'    : f"{ml_probas['XGBoost']:.1%}  {risk_label(ml_probas['XGBoost'])}",
                'Score Ensemble Final'       : f"{ensemble_proba:.1%}  {risk_label(ensemble_proba)}",
            },

            'SECTION_3_ALERTES': [
                f"[{r['id']}] P{r['priority']} {r['category']}: {r['message']}  →  {r['action']}"
                for r in expert_rules if r['priority'] <= 2
            ] or ['✅ Aucune alerte clinique urgente'],

            'SECTION_4_ALIMENTATION': {
                'Recommandations Profil' : profil_data['alimentation'],
                'Via Patients Similaires': collab_result['recs'],
            },

            'SECTION_5_PROBIOTIQUES': profil_data['probiotiques'],

            'SECTION_6_LIFESTYLE': profil_data['lifestyle'],

            'SECTION_7_SUIVI': {
                'Plan de suivi'         : profil_data['suivi'],
                'Règles de bonne vie'   : [
                    r['action'] for r in expert_rules if r['priority'] == 3
                ],
                'Prochain kit recommandé': profil_data['suivi'],
            },
        }


# ── Instanciation ─────────────────────────────────────────────────────
engine_unified = BiomeXRecommendationEngine(
    cluster_recs = CLUSTER_RECS,
    collab_rec   = collab,
    expert_engine= expert,
    ml_models    = trained,
    pca          = pca,
    n_pca        = n_comp90,
)

# ── Génération du rapport complet ────────────────────────────────────
TEST_IDX = 42
full_report = engine_unified.generate(
    patient_idx   = TEST_IDX,
    SC_df         = SC,
    df_clin_full  = df_clin,
    X_feat_full   = X_feat,
    feat_names_list= feat_names[:X_feat.shape[1]],
)

print(f"\n{'═'*65}")
print(f"  RAPPORT BIOMEX COMPLET — {full_report['patient_id']}")
print(f"  Date : {full_report['date']}")
print(f"{'═'*65}")

for section, content in full_report.items():
    if section in ('patient_id','date'):
        continue
    print(f"\n{'─'*65}")
    print(f"  {section}")
    print(f"{'─'*65}")
    if isinstance(content, dict):
        for k, v in content.items():
            if isinstance(v, list):
                print(f"  {k} :")
                for item in v: print(f"    • {item}")
            else:
                print(f"  {k:<40} : {v}")
    elif isinstance(content, list):
        for item in content: print(f"  • {item}")

print(f"\n{'═'*65}")

---
## 18. Rapport Visuel Patient & Sauvegarde <a id='18'></a>

In [ ]:
# ── Rapport visuel complet ────────────────────────────────────────────
def render_full_report(patient_idx, full_report, SC, X_umap, CLUSTER_KM,
                        PROBA_GMM, K_GMM, save_path=None):

    fig = plt.figure(figsize=(22, 28))
    fig.patch.set_facecolor('#F0F7FF')
    gs  = gridspec.GridSpec(5, 3, figure=fig,
                             hspace=0.45, wspace=0.32,
                             top=0.94, bottom=0.04, left=0.05, right=0.97)

    pid   = full_report['patient_id']
    sc    = SC.iloc[patient_idx]

    # ─ Header ─────────────────────────────────────────────────────────
    ax_hd = fig.add_subplot(gs[0, :])
    ax_hd.set_facecolor(C['navy'])
    ax_hd.text(0.02, 0.68, 'BiomeX', color='#66CCFF', fontsize=36,
               fontweight='bold', transform=ax_hd.transAxes, fontfamily='monospace')
    ax_hd.text(0.22, 0.68, 'Rapport Microbiome Personnalisé', color='white',
               fontsize=18, transform=ax_hd.transAxes, va='center')
    ax_hd.text(0.02, 0.22, f'Patient : {pid}  |  Dakar, Sénégal  |  Février 2026',
               color='#AACCEE', fontsize=12, transform=ax_hd.transAxes)
    ax_hd.text(0.75, 0.45, f'Score Global : {sc["global"]:.0f}/100',
               color='#FFDD55', fontsize=22, fontweight='bold',
               transform=ax_hd.transAxes)
    ax_hd.axis('off')

    # ─ Gauge score global ─────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[1, 0])
    g   = sc['global']
    gc  = C['green'] if g>66 else C['orange'] if g>33 else C['red']
    theta = np.linspace(np.pi, 0, 200)
    ax1.plot(np.cos(theta), np.sin(theta), color='#DDDDDD', lw=20)
    ratio = g/100
    th2   = np.linspace(np.pi, np.pi-ratio*np.pi, 200)
    ax1.plot(np.cos(th2), np.sin(th2), color=gc, lw=20)
    ax1.text(0, 0.2, f'{g:.0f}', ha='center', fontsize=42, fontweight='bold', color=gc)
    ax1.text(0, -0.15, '/100', ha='center', fontsize=16, color='#888888')
    ax1.text(0, -0.45, 'Score Métabolique', ha='center', fontsize=10, color='#555555')
    ax1.set_xlim(-1.4,1.4); ax1.set_ylim(-0.6,1.2); ax1.axis('off')
    ax1.set_title('Score Global BiomeX', fontweight='bold', color=C['navy'])

    # ─ Radar des 5 scores ─────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 1], projection='polar')
    cats  = ['Diversité\nShannon','Anti-\nInflam.','SCFA','Anti-\nDysbiose','Global']
    vals  = [
        min(sc['H']/4.5, 1),
        max(0, 1-(sc['inflam']+3)/6),
        min(sc['scfa']*10, 1),
        max(0, 1-sc['dysbiose']/3),
        sc['global']/100,
    ]
    N = len(cats)
    ang = [n/N*2*np.pi for n in range(N)] + [0]
    vals += [vals[0]]
    ax2.set_theta_offset(np.pi/2); ax2.set_theta_direction(-1)
    ax2.plot(ang, vals, color=C['green'], lw=2)
    ax2.fill(ang, vals, alpha=0.25, color=C['green'])
    ax2.set_xticks(ang[:-1]); ax2.set_xticklabels(cats, size=9)
    ax2.set_ylim(0,1); ax2.set_yticks([])
    ax2.set_title('Profil Microbiomique', fontweight='bold', color=C['navy'], pad=20)

    # ─ Position dans la cohorte (UMAP) ────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 2])
    for k in range(K_OPTIMAL):
        m = CLUSTER_KM==k
        ax3.scatter(X_umap[m,0], X_umap[m,1], c=CLUSTER_PALETTE[k],
                    alpha=0.3, s=15)
    ax3.scatter(X_umap[patient_idx,0], X_umap[patient_idx,1],
                c='red', s=200, zorder=10, marker='★',
                edgecolors='white', lw=2, label=f'{pid}')
    ax3.legend(fontsize=10)
    ax3.set_title('Votre Position dans la Cohorte', fontweight='bold', color=C['navy'])
    ax3.set_xlabel('UMAP1'); ax3.set_ylabel('UMAP2')

    # ─ Risques ML ─────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    risk_data  = full_report['SECTION_2_RISQUES_ML']
    risk_labels= list(risk_data.keys())
    risk_vals  = [float(v.split('%')[0].replace(',','.')) for v in risk_data.values()]
    risk_cols  = [C['green'] if v<25 else C['orange'] if v<60 else C['red']
                  for v in risk_vals]
    bars = ax4.barh(risk_labels, risk_vals, color=risk_cols, edgecolor='white')
    ax4.axvline(25, color='orange', ls='--', lw=1, alpha=0.7)
    ax4.axvline(60, color='red',    ls='--', lw=1, alpha=0.7)
    ax4.set_xlim(0, 100)
    ax4.set_xlabel('Probabilité de risque (%)')
    ax4.set_title('Risques Prédits (ML)', fontweight='bold', color=C['navy'])
    for bar, val in zip(bars, risk_vals):
        ax4.text(val+1, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

    # ─ Probabilités GMM ───────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    gmm_probas = PROBA_GMM[patient_idx]
    ax5.bar(range(K_GMM), gmm_probas*100,
            color=CLUSTER_PALETTE[:K_GMM], edgecolor='white')
    ax5.set_xticks(range(K_GMM))
    ax5.set_xticklabels([f'Profil {k}' for k in range(K_GMM)], fontsize=9)
    ax5.set_ylabel('Probabilité (%)')
    ax5.set_title('Appartenance aux Profils GMM', fontweight='bold', color=C['navy'])
    ax5.set_ylim(0, 100)
    for i, v in enumerate(gmm_probas*100):
        ax5.text(i, v+1, f'{v:.1f}%', ha='center', fontsize=9)

    # ─ Top bactéries ──────────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[2, 2])
    taxa_p   = X_rel.iloc[patient_idx]
    top10    = taxa_p.nlargest(10)
    cols_bact= [C['red']   if b in PATHOBIONTES
                else C['green'] if b in BUTYRATE_PRODUCERS
                else C['blue'] for b in top10.index]
    ax6.barh(range(len(top10)), top10.values*100, color=cols_bact, edgecolor='white')
    ax6.set_yticks(range(len(top10)))
    ax6.set_yticklabels([b.replace('_',' ')[:22] for b in top10.index], fontsize=8)
    ax6.invert_yaxis()
    ax6.set_xlabel('Abondance (%)')
    ax6.set_title('Top 10 Espèces Bactériennes', fontweight='bold', color=C['navy'])
    patches_b = [mpatches.Patch(color=C['red'],  label='Pathobionte'),
                 mpatches.Patch(color=C['green'],label='Prod. SCFA'),
                 mpatches.Patch(color=C['blue'], label='Autre')]
    ax6.legend(handles=patches_b, fontsize=8, loc='lower right')

    # ─ Recommandations alimentation ───────────────────────────────────
    ax7 = fig.add_subplot(gs[3, :])
    ax7.set_facecolor(C['light'])
    ax7.set_xlim(0,1); ax7.set_ylim(0,1); ax7.axis('off')
    ax7.set_title('Recommandations Personnalisées', fontweight='bold',
                  color=C['navy'], fontsize=14, pad=12)

    recs_alim    = full_report['SECTION_4_ALIMENTATION']['Recommandations Profil'][:4]
    recs_collab  = full_report['SECTION_4_ALIMENTATION']['Via Patients Similaires'][:4]
    recs_probio  = full_report['SECTION_5_PROBIOTIQUES'][:4]

    for col_i, (title_txt, recs, col) in enumerate([
        ('🥗 Alimentation (profil)', recs_alim,   C['green']),
        ('👥 Patients similaires',   recs_collab, C['blue']),
        ('🦠 Probiotiques',          recs_probio, C['teal']),
    ]):
        xs = col_i * 0.34 + 0.01
        ax7.text(xs, 0.92, title_txt, fontsize=11, fontweight='bold',
                 color=col, transform=ax7.transAxes)
        for j, r in enumerate(recs):
            ax7.text(xs, 0.78 - j*0.19,
                     f'• {str(r)[:68]}',
                     fontsize=8.5, color='#333333', transform=ax7.transAxes)

    # ─ Alertes + footer ───────────────────────────────────────────────
    ax8 = fig.add_subplot(gs[4, :])
    ax8.set_facecolor('#FFF3E0' if full_report['SECTION_3_ALERTES'][0]!='✅ Aucune alerte clinique urgente'
                       else '#E8F5E9')
    ax8.set_xlim(0,1); ax8.set_ylim(0,1); ax8.axis('off')

    alertes_str = '  ·  '.join(full_report['SECTION_3_ALERTES'][:2])
    ax8.text(0.5, 0.68, f'Alertes : {alertes_str}',
             ha='center', fontsize=9,
             color=C['red'] if '🔴' in alertes_str else '#333333',
             fontweight='bold', transform=ax8.transAxes)
    ax8.text(0.5, 0.28,
             '📋 Ce rapport est un outil d\'aide à la décision. Il ne remplace pas l\'avis médical.  '
             '|  BiomeX © 2026  |  Dakar, Sénégal  |  abdou.ba@biomex.ai',
             ha='center', fontsize=8.5, color='#888888',
             style='italic', transform=ax8.transAxes)

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f"💾 Rapport sauvegardé : {save_path}")
    plt.show()


render_full_report(
    patient_idx = TEST_IDX,
    full_report = full_report,
    SC          = SC,
    X_umap      = X_umap,
    CLUSTER_KM  = CLUSTER_KM,
    PROBA_GMM   = PROBA_GMM,
    K_GMM       = K_GMM,
    save_path   = 'BiomeX_Rapport_Complet.png'
)

In [ ]:
# ── Sauvegarde de tous les modèles ────────────────────────────────────
import pickle, os
os.makedirs('models', exist_ok=True)

to_save = {
    'rf_model.pkl'   : trained['RandomForest'],
    'xgb_model.pkl'  : trained['XGBoost'],
    'lr_model.pkl'   : trained['Logistique'],
    'pca.pkl'        : pca,
    'kmeans.pkl'     : kmeans,
    'gmm.pkl'        : gmm_final,
    'dbscan.pkl'     : dbscan,
    'ae_model.pt'    : None,  # PyTorch sauvegarde séparée
}

for fname, obj in to_save.items():
    if obj is not None:
        with open(f'models/{fname}', 'wb') as f:
            pickle.dump(obj, f)

torch.save(ae_model.state_dict(), 'models/autoencoder.pt')
SC.to_csv('cohorte_scores_biomex.csv')

print("\n" + "═"*65)
print("  ✅ PIPELINE BIOMEX v5.0 — COMPLET")
print("═"*65)

summary = [
    ('DONNÉES',         f'{N_PATIENTS} patients · {N_BACTERIA} espèces · {N_FUNCTIONS} fonctions'),
    ('PREPROCESSING',   'CLR normalization · Matrice globale unifiée'),
    ('RÉDUCTION',       f'PCA ({n_comp90} CP, 90% var.) · UMAP 2D · Autoencoder (latent=32)'),
    ('SCORES',          'Shannon H · Inflammation · SCFA · Dysbiose · Global'),
    ('NON SUPERVISÉ',   f'K-Means (K={K_OPTIMAL}) · Ward · GMM (K={K_GMM}) · DBSCAN · Co-abondance'),
    ('SUPERVISÉ',       'Logistic · RandomForest · XGBoost · K-Fold · SHAP'),
    ('RECOMMANDATIONS', 'Cluster-based · Collaboratif (KNN) · Règles expertes (12 règles) · ML'),
    ('RAPPORT',         'Dashboard visuel 5 panels · Export PNG · Modèles sérialisés'),
]
for cat, detail in summary:
    print(f"  ✅ {cat:<20} → {detail}")
print()